# CyberHackAI — V13 Hybrid URL + HTML, standalone

Полностью новая версия. Она **не использует** старые notebook, PKL, OOF или промежуточные CSV.

Нужны только исходные файлы:

- `train.csv`
- `test.csv`
- `sample_submission.csv`
- `train_with_content.csv`
- `test_with_content.csv`

### Новая архитектура

1. URL-модель по нормализованным char n-grams.
2. Отдельные host и path/query модели.
3. Числовые URL-признаки.
4. Потоковое извлечение структурных HTML-признаков из огромных `*_with_content.csv`.
5. Title-модель по содержимому `<title>`.
6. HTML structural model: формы, password-inputs, iframe, redirects, внешние ссылки, JS-obfuscation и др.
7. URL-only OOF stack и Hybrid OOF stack сравниваются честно.
8. HTML используется в финальном submission **только если OOF показывает улучшение**.
9. Для второго решения используется segment routing: HTML применяется только к строкам, где content реально найден и полезен.

Большие content CSV читаются chunks и целиком в RAM не загружаются.


In [ ]:
# 0. Imports/config
import os, sys, re, gc, time, html as html_lib, pickle, warnings
from pathlib import Path
from urllib.parse import urlsplit, unquote

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve

warnings.filterwarnings('ignore')

SEED = 2026
N_SPLITS = 5
CHUNK_SIZE = 2500
FAST_MODE = False
REBUILD_CONTENT_CACHE = False
TRAIN_CONTENT_CACHE = Path('v13_train_content_features.pkl')
TEST_CONTENT_CACHE = Path('v13_test_content_features.pkl')

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('CPU:', os.cpu_count())


In [ ]:
# 1. Load source files only
TRAIN_PATH = Path('train.csv')
TEST_PATH = Path('test.csv')
SAMPLE_PATH = Path('sample_submission.csv')
TRAIN_CONTENT_PATH = Path('train_with_content.csv')
TEST_CONTENT_PATH = Path('test_with_content.csv')

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_PATH, TRAIN_CONTENT_PATH, TEST_CONTENT_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Не найден обязательный файл: {p}')

train_raw = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

TARGET = 'Predicted' if 'Predicted' in train_raw.columns else 'result'
if TARGET not in train_raw.columns:
    raise ValueError('В train.csv не найден target: Predicted/result')

assert 'url' in train_raw.columns and 'url' in test.columns
assert 'Id' in test.columns and 'Id' in sample.columns

train_raw['url'] = train_raw['url'].fillna('').astype(str)
test['url'] = test['url'].fillna('').astype(str)
train_raw[TARGET] = pd.to_numeric(train_raw[TARGET], errors='raise').astype(int)

print('train:', train_raw.shape)
print('test:', test.shape)
print('sample:', sample.shape)
print(train_raw[TARGET].value_counts().sort_index())
print('phishing share:', float(train_raw[TARGET].mean()))


In [ ]:
# 2. Canonical URL + clean train

def canonical_url_key(v):
    if pd.isna(v):
        return ''
    s = html_lib.unescape(str(v)).strip().strip('"').strip("'")
    try:
        s = unquote(s)
    except Exception:
        pass
    c = s if '://' in s else 'http://' + s
    try:
        p = urlsplit(c)
        host = (p.hostname or '').lower().strip('.')
        if host.startswith('www.'):
            host = host[4:]
        path = re.sub(r'/+', '/', p.path or '/')
        if path != '/':
            path = path.rstrip('/')
        query = p.query or ''
        return (host + path + (('?' + query) if query else '')).lower()
    except Exception:
        s = re.sub(r'^https?://', '', s, flags=re.I)
        s = re.sub(r'^www\.', '', s, flags=re.I)
        return s.rstrip('/').lower()

label_nunique = train_raw.groupby('url', sort=False)[TARGET].nunique()
conflicting_urls = set(label_nunique[label_nunique > 1].index)
train = train_raw.loc[~train_raw['url'].isin(conflicting_urls)].drop_duplicates('url').reset_index(drop=True)
y = train[TARGET].to_numpy(np.int8)

exact_lookup = (
    train_raw.groupby('url', sort=False)[TARGET]
    .agg(lambda s: int(s.iloc[0]) if s.nunique() == 1 else np.nan)
    .dropna().astype(int).to_dict()
)

print('conflicting:', len(conflicting_urls))
print('clean train:', train.shape)
print('clean phishing share:', float(y.mean()))


In [ ]:
# 3. URL parser

def normalize_dynamic(s):
    s = str(s).lower()
    s = re.sub(r'[0-9a-f]{12,}', '{hex}', s)
    s = re.sub(r'\d{4,}', '{num}', s)
    return s

def parse_url(v):
    raw = str(v).strip()
    c = raw if '://' in raw else 'http://' + raw
    try:
        p = urlsplit(c)
        host = (p.hostname or '').lower().strip('.')
        path = (p.path or '').lower()
        query = (p.query or '').lower()
        scheme = (p.scheme or '').lower()
    except Exception:
        host = path = query = scheme = ''
    labels = [x for x in host.split('.') if x]
    root = '.'.join(labels[-2:]) if len(labels) >= 2 else host
    subdomain = '.'.join(labels[:-2]) if len(labels) > 2 else ''
    normalized = host + normalize_dynamic(path) + (('?' + normalize_dynamic(query)) if query else '')
    return {
        'host': host, 'root': root, 'subdomain': subdomain,
        'path': path, 'query': query, 'scheme': scheme,
        'normalized': normalized, 'canonical': canonical_url_key(raw)
    }

parts_train = pd.DataFrame([parse_url(x) for x in train['url']])
parts_test = pd.DataFrame([parse_url(x) for x in test['url']])

for col in ['host','root','canonical']:
    known = parts_test[col].isin(set(parts_train[col]))
    print(col, 'known in train:', round(float(known.mean()), 4))


## 4. EDA, ориентированный на переносимость

Проверяем повторяемость host/root, одноклассовые доменные группы и exact train/test overlap. Это важнее обычной диаграммы длин URL, потому что random split может сильно переоценивать качество.


In [ ]:
# 4. Domain/campaign EDA
eda = pd.DataFrame({'y': y, 'host': parts_train['host'], 'root': parts_train['root']})
for col in ['host','root']:
    st = eda.groupby(col)['y'].agg(['count','mean','nunique'])
    repeated_share = st.loc[st['count'] > 1, 'count'].sum() / len(eda)
    pure_share = (st['nunique'] == 1).mean()
    print(col, '| groups:', len(st), '| repeated rows:', round(float(repeated_share),4), '| pure groups:', round(float(pure_share),4))

overlap = test['url'].isin(exact_lookup)
print('Exact train/test overlap:', int(overlap.sum()))
if overlap.any():
    known_labels = test.loc[overlap, 'url'].map(exact_lookup).astype(int)
    print('Exact overlap phishing share:', round(float(known_labels.mean()), 6))


In [ ]:
# 5. URL numeric features
SUSP_WORDS = ['login','signin','verify','secure','account','password','payment','billing','update','confirm','recover','unlock','wallet','invoice','support','auth','bank','bonus','gift','free','redirect','webscr']
BRANDS = ['paypal','apple','microsoft','google','facebook','instagram','amazon','netflix','visa','mastercard','outlook','office','icloud','whatsapp','telegram','linkedin','dropbox','adobe','steam','binance']

def entropy_str(s):
    s = str(s)
    if not s:
        return 0.0
    _, c = np.unique(list(s), return_counts=True)
    p = c / c.sum()
    return float(-(p * np.log2(p)).sum())

def build_url_numeric(urls, parts):
    urls = pd.Series(urls).fillna('').astype(str).reset_index(drop=True)
    low = urls.str.lower()
    out = pd.DataFrame(index=np.arange(len(urls)))
    out['url_len'] = urls.str.len()
    out['host_len'] = parts['host'].str.len().to_numpy()
    out['path_len'] = parts['path'].str.len().to_numpy()
    out['query_len'] = parts['query'].str.len().to_numpy()
    out['dots'] = urls.str.count(r'\.')
    out['dashes'] = urls.str.count('-')
    out['digits'] = urls.str.count(r'\d')
    out['ats'] = urls.str.count('@')
    out['percents'] = urls.str.count('%')
    out['equals'] = urls.str.count('=')
    out['amp'] = urls.str.count('&')
    out['subdomain_depth'] = (parts['subdomain'].str.count(r'\.') + (parts['subdomain'] != '').astype(int)).to_numpy()
    out['path_depth'] = parts['path'].str.count('/').to_numpy()
    out['is_ip'] = parts['host'].str.match(r'^(?:\d{1,3}\.){3}\d{1,3}$').astype(int).to_numpy()
    out['punycode'] = parts['host'].str.contains('xn--', regex=False).astype(int).to_numpy()
    out['https_token_in_host'] = parts['host'].str.contains('https', regex=False).astype(int).to_numpy()
    out['uses_https'] = (parts['scheme'] == 'https').astype(int).to_numpy()
    out['suspicious_hits'] = 0
    out['brand_hits'] = 0
    for w in SUSP_WORDS:
        out['suspicious_hits'] += low.str.contains(w, regex=False).astype(int)
    for b in BRANDS:
        out['brand_hits'] += low.str.contains(b, regex=False).astype(int)
    out['entropy_url'] = [entropy_str(x) for x in urls]
    out['entropy_host'] = [entropy_str(x) for x in parts['host']]
    out['entropy_path'] = [entropy_str(x) for x in parts['path']]
    return out.astype(np.float32)

url_num_train = build_url_numeric(train['url'], parts_train)
url_num_test = build_url_numeric(test['url'], parts_test)
display(url_num_train.describe().T)


## 6. Потоковое извлечение HTML-признаков

Вместо обучения на полном тексте гигабайтных HTML-файлов извлекается компактный набор структурных сигналов: формы, password inputs, iframe, redirects, внешние ссылки, JS-obfuscation, title и brand mismatch.


In [ ]:
# 6.1 Infer content CSV schema

def infer_content_schema(path):
    preview = pd.read_csv(path, nrows=5, low_memory=False)
    cols = list(preview.columns)
    lc = {c: str(c).lower() for c in cols}
    url_candidates = [c for c in cols if lc[c] in {'url','uri','link'}] or [c for c in cols if 'url' in lc[c]]
    if not url_candidates:
        raise ValueError(f'URL column not found in {path}: {cols}')
    id_candidates = [c for c in cols if lc[c] in {'id','index'}]
    target_candidates = [c for c in cols if lc[c] in {'predicted','result','label','target','class','y'}]
    service = {url_candidates[0]}
    if id_candidates: service.add(id_candidates[0])
    if target_candidates: service.add(target_candidates[0])
    content_cols = [c for c in cols if c not in service and any(k in lc[c] for k in ['content','html','body','page','source','text'])]
    if not content_cols:
        content_cols = [c for c in cols if c not in service and preview[c].dtype == 'object']
    return {'url':url_candidates[0], 'id':id_candidates[0] if id_candidates else None, 'target':target_candidates[0] if target_candidates else None, 'content_cols':content_cols, 'columns':cols}

schema_ct_train = infer_content_schema(TRAIN_CONTENT_PATH)
schema_ct_test = infer_content_schema(TEST_CONTENT_PATH)
print('train_with_content:', schema_ct_train)
print('test_with_content :', schema_ct_test)


In [ ]:
# 6.2 HTML feature extractor
RE_TITLE = re.compile(r'(?is)<title[^>]*>(.*?)</title>')
RE_FORM = re.compile(r'(?is)<form\b[^>]*>')
RE_FORM_ACTION = re.compile(r'(?is)\baction\s*=\s*["\']?([^"\' >]+)')
RE_INPUT = re.compile(r'(?is)<input\b[^>]*>')
RE_PASSWORD = re.compile(r'(?is)<input\b[^>]*\btype\s*=\s*["\']?password\b')
RE_HIDDEN = re.compile(r'(?is)<input\b[^>]*\btype\s*=\s*["\']?hidden\b')
RE_IFRAME = re.compile(r'(?is)<iframe\b')
RE_SCRIPT = re.compile(r'(?is)<script\b')
RE_ANCHOR = re.compile(r'(?is)<a\b[^>]*\bhref\s*=\s*["\']([^"\']+)["\']')
RE_META_REFRESH = re.compile(r'(?is)<meta\b[^>]*http-equiv\s*=\s*["\']?refresh')
RE_TAG = re.compile(r'(?s)<[^>]+>')
RE_SPACE = re.compile(r'\s+')

def same_host_or_relative(link, page_host):
    link = str(link).strip()
    if not link or link.startswith(('#','/','javascript:','mailto:','tel:')):
        return True
    try:
        p = urlsplit(link if '://' in link else 'http://' + link)
        h = (p.hostname or '').lower().strip('.')
        if h.startswith('www.'): h = h[4:]
        ph = page_host[4:] if page_host.startswith('www.') else page_host
        return (not h) or h == ph or h.endswith('.' + ph)
    except Exception:
        return True

def extract_html_features(raw_html, page_host):
    s = '' if pd.isna(raw_html) else str(raw_html)
    low = s.lower()
    m = RE_TITLE.search(s)
    title = html_lib.unescape(RE_TAG.sub(' ', m.group(1))) if m else ''
    title = RE_SPACE.sub(' ', title).strip()[:500]
    forms = RE_FORM.findall(s)
    form_actions = RE_FORM_ACTION.findall(' '.join(forms))
    anchors = RE_ANCHOR.findall(s)
    external_links = sum(0 if same_host_or_relative(h, page_host) else 1 for h in anchors[:1000])
    external_form_actions = sum(0 if same_host_or_relative(a, page_host) else 1 for a in form_actions[:100])
    visible = html_lib.unescape(RE_TAG.sub(' ', s[:120000])).lower()
    suspicious_hits = sum(visible.count(w) for w in SUSP_WORDS)
    brand_hits = sum(visible.count(b) for b in BRANDS)
    title_low = title.lower()
    title_brands = [b for b in BRANDS if b in title_low]
    title_brand_mismatch = int(bool(title_brands) and not any(b in page_host.lower() for b in title_brands))
    return {
        'content_present': int(bool(s.strip())),
        'html_len': len(s),
        'title_len': len(title),
        'form_count': len(forms),
        'input_count': len(RE_INPUT.findall(s)),
        'password_count': len(RE_PASSWORD.findall(s)),
        'hidden_input_count': len(RE_HIDDEN.findall(s)),
        'iframe_count': len(RE_IFRAME.findall(s)),
        'script_count': len(RE_SCRIPT.findall(s)),
        'anchor_count': len(anchors),
        'external_link_count': external_links,
        'external_link_ratio': external_links / max(len(anchors), 1),
        'external_form_action_count': external_form_actions,
        'meta_refresh': int(bool(RE_META_REFRESH.search(s))),
        'js_location': int(('window.location' in low) or ('document.location' in low) or ('location.href' in low)),
        'js_eval': int('eval(' in low),
        'js_atob': int('atob(' in low),
        'js_fromcharcode': int('fromcharcode' in low),
        'mailto_count': low.count('mailto:'),
        'data_uri_count': low.count('data:'),
        'suspicious_text_hits': suspicious_hits,
        'brand_text_hits': brand_hits,
        'title_brand_hits': len(title_brands),
        'title_brand_mismatch': title_brand_mismatch,
        'html_entropy_sample': entropy_str(s[:20000]),
        'title': title,
    }

HTML_NUMERIC_COLS = ['content_present','html_len','title_len','form_count','input_count','password_count','hidden_input_count','iframe_count','script_count','anchor_count','external_link_count','external_link_ratio','external_form_action_count','meta_refresh','js_location','js_eval','js_atob','js_fromcharcode','mailto_count','data_uri_count','suspicious_text_hits','brand_text_hits','title_brand_hits','title_brand_mismatch','html_entropy_sample']


In [ ]:
# 6.3 Streaming extraction with Id/canonical matching

def _build_lookup(official_df):
    by_id = None
    if 'Id' in official_df.columns:
        by_id = pd.Series(np.arange(len(official_df)), index=official_df['Id'].astype(str))
        by_id = by_id[~by_id.index.duplicated(keep='first')]
    by_key = pd.Series(np.arange(len(official_df)), index=official_df['url'].astype(str).map(canonical_url_key))
    by_key = by_key[~by_key.index.duplicated(keep='first')]
    return by_id, by_key

def extract_content_features_stream(path, schema, official_df, official_parts, cache_path):
    if cache_path.exists() and not REBUILD_CONTENT_CACHE:
        obj = pd.read_pickle(cache_path)
        if len(obj) == len(official_df):
            print('Loaded cache:', cache_path)
            return obj
    by_id, by_key = _build_lookup(official_df)
    rows = [None] * len(official_df)
    matched = np.zeros(len(official_df), dtype=bool)
    usecols = list(dict.fromkeys([schema['url']] + ([schema['id']] if schema['id'] else []) + schema['content_cols']))
    for chunk_no, chunk in enumerate(pd.read_csv(path, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False), 1):
        idx = np.full(len(chunk), -1, dtype=np.int64)
        if schema['id'] and by_id is not None:
            m = chunk[schema['id']].astype(str).map(by_id)
            ok = m.notna().to_numpy()
            idx[ok] = m.loc[ok].astype(int).to_numpy()
        need = idx < 0
        if need.any():
            keys = chunk.loc[need, schema['url']].fillna('').astype(str).map(canonical_url_key)
            m2 = keys.map(by_key)
            ok2 = m2.notna().to_numpy()
            pos = np.flatnonzero(need)
            idx[pos[ok2]] = m2.loc[ok2].astype(int).to_numpy()
        for pos in np.flatnonzero(idx >= 0):
            oi = int(idx[pos])
            if matched[oi]:
                continue
            raw_html = ' '.join('' if pd.isna(chunk.iloc[pos][c]) else str(chunk.iloc[pos][c]) for c in schema['content_cols'])
            rows[oi] = extract_html_features(raw_html, str(official_parts.iloc[oi]['host']))
            matched[oi] = True
        if chunk_no % 100 == 0:
            print(path.name, 'chunk', chunk_no, 'matched', int(matched.sum()), '/', len(matched))
        del chunk
        gc.collect()
    zero = {c: 0 for c in HTML_NUMERIC_COLS}; zero['title'] = ''
    result = pd.DataFrame([rows[i] if rows[i] is not None else zero.copy() for i in range(len(rows))])
    result['matched_content'] = matched.astype(np.int8)
    result.to_pickle(cache_path)
    print(path.name, 'coverage:', round(float(matched.mean()), 6))
    return result

content_train = extract_content_features_stream(TRAIN_CONTENT_PATH, schema_ct_train, train, parts_train, TRAIN_CONTENT_CACHE)
content_test = extract_content_features_stream(TEST_CONTENT_PATH, schema_ct_test, test, parts_test, TEST_CONTENT_CACHE)
print('train content coverage:', float(content_train['matched_content'].mean()))
print('test content coverage:', float(content_test['matched_content'].mean()))


In [ ]:
# 7. Content EDA
ct_eda = content_train[HTML_NUMERIC_COLS + ['matched_content']].copy()
ct_eda['y'] = y
print('matched:', int(content_train['matched_content'].sum()), 'missing:', int((content_train['matched_content'] == 0).sum()))
if content_train['matched_content'].sum() > 0:
    m = ct_eda[ct_eda['matched_content'] == 1]
    corr = m.corr(numeric_only=True)['y'].drop('y').sort_values(key=lambda s: s.abs(), ascending=False)
    display(corr.head(20).to_frame('corr_with_target'))
    display(m.groupby('y')[['html_len','form_count','password_count','external_link_ratio','meta_refresh','js_location','suspicious_text_hits','title_brand_mismatch']].mean())


In [ ]:
# 8. Base vectorizers
url_vec = HashingVectorizer(analyzer='char', ngram_range=(3,6), n_features=2**18, alternate_sign=False, norm=None, lowercase=True, dtype=np.float32)
host_vec = HashingVectorizer(analyzer='char', ngram_range=(2,5), n_features=2**15, alternate_sign=False, norm=None, lowercase=True, dtype=np.float32)
path_vec = HashingVectorizer(analyzer='char', ngram_range=(3,5), n_features=2**16, alternate_sign=False, norm=None, lowercase=True, dtype=np.float32)
title_vec = HashingVectorizer(analyzer='char', ngram_range=(2,5), n_features=2**15, alternate_sign=False, norm='l2', lowercase=True, dtype=np.float32)
BASE_NAMES = ['url_char','host_char','path_char','url_numeric','title_char','html_struct','content_available']


In [ ]:
# 9. Base OOF + test inference
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros((len(train), len(BASE_NAMES)), dtype=np.float64)
test_base = np.zeros((len(test), len(BASE_NAMES)), dtype=np.float64)

X_url = parts_train['normalized'].astype(str).to_numpy(); T_url = parts_test['normalized'].astype(str).to_numpy()
X_host = parts_train['host'].astype(str).to_numpy(); T_host = parts_test['host'].astype(str).to_numpy()
X_path = (parts_train['path'] + '?' + parts_train['query']).astype(str).to_numpy(); T_path = (parts_test['path'] + '?' + parts_test['query']).astype(str).to_numpy()
title_train = content_train['title'].fillna('').astype(str).to_numpy(); title_test = content_test['title'].fillna('').astype(str).to_numpy()
html_num_train = content_train[HTML_NUMERIC_COLS].astype(np.float32); html_num_test = content_test[HTML_NUMERIC_COLS].astype(np.float32)
train_has_content = content_train['matched_content'].to_numpy() == 1
test_has_content = content_test['matched_content'].to_numpy() == 1

for fold, (tr, va) in enumerate(skf.split(train, y), 1):
    t0 = time.time()
    A = url_vec.transform(X_url[tr]); B = url_vec.transform(X_url[va]); TT = url_vec.transform(T_url)
    tf = TfidfTransformer(sublinear_tf=True); A = tf.fit_transform(A); B = tf.transform(B); TT = tf.transform(TT)
    m_url = LinearSVC(C=0.7); m_url.fit(A, y[tr])
    oof[va,0] = m_url.decision_function(B); test_base[:,0] += m_url.decision_function(TT)/N_SPLITS

    Ah = host_vec.transform(X_host[tr]); Bh = host_vec.transform(X_host[va]); Th = host_vec.transform(T_host)
    tfh = TfidfTransformer(sublinear_tf=True); Ah = tfh.fit_transform(Ah); Bh = tfh.transform(Bh); Th = tfh.transform(Th)
    m_host = SGDClassifier(loss='log_loss', alpha=3e-6, average=True, max_iter=1600, tol=1e-4, random_state=SEED+fold)
    m_host.fit(Ah, y[tr]); oof[va,1] = m_host.predict_proba(Bh)[:,1]; test_base[:,1] += m_host.predict_proba(Th)[:,1]/N_SPLITS

    Ap = path_vec.transform(X_path[tr]); Bp = path_vec.transform(X_path[va]); Tp = path_vec.transform(T_path)
    tfp = TfidfTransformer(sublinear_tf=True); Ap = tfp.fit_transform(Ap); Bp = tfp.transform(Bp); Tp = tfp.transform(Tp)
    m_path = SGDClassifier(loss='log_loss', alpha=4e-6, average=True, max_iter=1600, tol=1e-4, random_state=SEED+100+fold)
    m_path.fit(Ap, y[tr]); oof[va,2] = m_path.predict_proba(Bp)[:,1]; test_base[:,2] += m_path.predict_proba(Tp)[:,1]/N_SPLITS

    m_num = HistGradientBoostingClassifier(learning_rate=0.06, max_iter=350 if not FAST_MODE else 80, max_leaf_nodes=31, l2_regularization=3.0, random_state=SEED+200+fold)
    m_num.fit(url_num_train.iloc[tr], y[tr]); oof[va,3] = m_num.predict_proba(url_num_train.iloc[va])[:,1]; test_base[:,3] += m_num.predict_proba(url_num_test)[:,1]/N_SPLITS

    trc = tr[train_has_content[tr]]; vac = va[train_has_content[va]]
    oof[va,4] = 0.5; oof[va,5] = 0.5
    test_base[:,4] += 0.5/N_SPLITS; test_base[:,5] += 0.5/N_SPLITS

    if len(trc) >= 200 and len(np.unique(y[trc])) == 2:
        mt = SGDClassifier(loss='log_loss', alpha=8e-6, average=True, max_iter=1200, tol=1e-4, random_state=SEED+300+fold)
        mt.fit(title_vec.transform(title_train[trc]), y[trc])
        if len(vac): oof[vac,4] = mt.predict_proba(title_vec.transform(title_train[vac]))[:,1]
        if test_has_content.any():
            test_base[test_has_content,4] += (mt.predict_proba(title_vec.transform(title_test[test_has_content]))[:,1] - 0.5)/N_SPLITS

        mh = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300 if not FAST_MODE else 60, max_leaf_nodes=25, l2_regularization=4.0, random_state=SEED+400+fold)
        mh.fit(html_num_train.iloc[trc], y[trc])
        if len(vac): oof[vac,5] = mh.predict_proba(html_num_train.iloc[vac])[:,1]
        if test_has_content.any():
            test_base[test_has_content,5] += (mh.predict_proba(html_num_test.iloc[np.flatnonzero(test_has_content)])[:,1] - 0.5)/N_SPLITS

    oof[va,6] = train_has_content[va].astype(float)
    test_base[:,6] = test_has_content.astype(float)
    print(f'fold {fold}/{N_SPLITS}: {time.time()-t0:.1f}s')
    gc.collect()


In [ ]:
# 10. Metrics helper + base quality

def best_threshold(y_true, score):
    p, r, t = precision_recall_curve(y_true, score)
    f = 2*p*r/(p+r+1e-15)
    if len(t) == 0: return 0.5, float(f[0])
    i = int(np.nanargmax(f[:-1]))
    return float(t[i]), float(f[i])

rows=[]
for j,name in enumerate(BASE_NAMES[:-1]):
    thr,f = best_threshold(y,oof[:,j]); pred=oof[:,j]>=thr
    rows.append({'model':name,'F1':f,'precision':precision_score(y,pred),'recall':recall_score(y,pred),'threshold':thr})
display(pd.DataFrame(rows).sort_values('F1',ascending=False))


In [ ]:
# 11. URL-only OOF stack
URL_COLS=[0,1,2,3]
meta_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED+1000)
url_oof=np.zeros(len(train),dtype=float)
for fold,(tr,va) in enumerate(meta_cv.split(oof,y),1):
    m=LogisticRegression(C=0.7,max_iter=2500,random_state=SEED+1100+fold)
    m.fit(oof[tr][:,URL_COLS],y[tr]); url_oof[va]=m.predict_proba(oof[va][:,URL_COLS])[:,1]
url_thr,url_f1=best_threshold(y,url_oof); url_pred=(url_oof>=url_thr).astype(int)
url_final=LogisticRegression(C=0.7,max_iter=2500,random_state=SEED+1200).fit(oof[:,URL_COLS],y)
url_test_score=url_final.predict_proba(test_base[:,URL_COLS])[:,1]
print('URL-only OOF F1:',url_f1,'precision:',precision_score(y,url_pred),'recall:',recall_score(y,url_pred),'threshold:',url_thr)


In [ ]:
# 12. Hybrid OOF stack
HYBRID_COLS=list(range(len(BASE_NAMES)))
hyb_oof=np.zeros(len(train),dtype=float)
for fold,(tr,va) in enumerate(meta_cv.split(oof,y),1):
    m=LogisticRegression(C=0.5,max_iter=2500,random_state=SEED+1300+fold)
    m.fit(oof[tr][:,HYBRID_COLS],y[tr]); hyb_oof[va]=m.predict_proba(oof[va][:,HYBRID_COLS])[:,1]
hyb_thr,hyb_f1=best_threshold(y,hyb_oof); hyb_pred=(hyb_oof>=hyb_thr).astype(int)
hyb_final=LogisticRegression(C=0.5,max_iter=2500,random_state=SEED+1400).fit(oof[:,HYBRID_COLS],y)
hyb_test_score=hyb_final.predict_proba(test_base[:,HYBRID_COLS])[:,1]
print('Hybrid OOF F1:',hyb_f1,'precision:',precision_score(y,hyb_pred),'recall:',recall_score(y,hyb_pred),'threshold:',hyb_thr,'delta:',hyb_f1-url_f1)


In [ ]:
# 13. Covered subset + group robustness diagnostic
covered=train_has_content
if covered.sum()>=500:
    url_thr_cov,url_f1_cov=best_threshold(y[covered],url_oof[covered])
    hyb_thr_cov,hyb_f1_cov=best_threshold(y[covered],hyb_oof[covered])
else:
    url_f1_cov=url_f1; hyb_f1_cov=-1.0; hyb_thr_cov=hyb_thr
print('covered rows:',int(covered.sum()),'URL F1:',url_f1_cov,'Hybrid F1:',hyb_f1_cov)

groups=parts_train['root'].astype(str).to_numpy(); url_g=[]; hyb_g=[]
try:
    gcv=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED+2000)
    for tr,va in gcv.split(oof,y,groups):
        mu=LogisticRegression(C=0.7,max_iter=2000).fit(oof[tr][:,URL_COLS],y[tr])
        mh=LogisticRegression(C=0.5,max_iter=2000).fit(oof[tr][:,HYBRID_COLS],y[tr])
        tu,_=best_threshold(y[tr],mu.predict_proba(oof[tr][:,URL_COLS])[:,1]); th,_=best_threshold(y[tr],mh.predict_proba(oof[tr][:,HYBRID_COLS])[:,1])
        url_g.append(f1_score(y[va],mu.predict_proba(oof[va][:,URL_COLS])[:,1]>=tu))
        hyb_g.append(f1_score(y[va],mh.predict_proba(oof[va][:,HYBRID_COLS])[:,1]>=th))
    print('URL grouped F1:',float(np.mean(url_g)),'Hybrid grouped F1:',float(np.mean(hyb_g)))
except Exception as e:
    print('Grouped diagnostic skipped:',repr(e))


In [ ]:
# 14. Safety gate
all_ok = hyb_f1 >= url_f1 + 0.0005
covered_ok = (covered.sum()<500) or (hyb_f1_cov >= url_f1_cov)
group_ok = True if not url_g else (np.mean(hyb_g) >= np.mean(url_g)-0.003)
USE_HYBRID_GLOBAL = bool(all_ok and covered_ok and group_ok)
USE_HYBRID_ROUTED = bool(covered.sum()>=500 and hyb_f1_cov >= url_f1_cov + 0.0005)
print('all_ok:',all_ok,'covered_ok:',covered_ok,'group_ok:',group_ok)
print('USE_HYBRID_GLOBAL:',USE_HYBRID_GLOBAL)
print('USE_HYBRID_ROUTED:',USE_HYBRID_ROUTED)


In [ ]:
# 15. Final predictions
url_pred_test=(url_test_score>=url_thr).astype(np.int8)
final_safe=(hyb_test_score>=hyb_thr).astype(np.int8) if USE_HYBRID_GLOBAL else url_pred_test.copy()
final_routed=url_pred_test.copy()
if USE_HYBRID_ROUTED and test_has_content.any():
    final_routed[test_has_content]=(hyb_test_score[test_has_content]>=hyb_thr_cov).astype(np.int8)

exact_mask=test['url'].isin(exact_lookup).to_numpy()
for i in np.flatnonzero(exact_mask):
    v=int(exact_lookup[test.iloc[i]['url']]); final_safe[i]=v; final_routed[i]=v

print('test content coverage:',float(test_has_content.mean()))
print('safe phishing share:',float(final_safe.mean()))
print('routed phishing share:',float(final_routed.mean()))
print('disagreement:',float(np.mean(final_safe!=final_routed)))


In [ ]:
# 16. Submissions + report
sub_safe=pd.DataFrame({'Id':test['Id'].to_numpy(copy=True),'Predicted':final_safe.astype(int)})
sub_routed=pd.DataFrame({'Id':test['Id'].to_numpy(copy=True),'Predicted':final_routed.astype(int)})
for sub in [sub_safe,sub_routed]:
    assert sub.columns.tolist()==['Id','Predicted']
    assert len(sub)==len(test)==len(sample)
    assert sub['Id'].reset_index(drop=True).equals(sample['Id'].reset_index(drop=True))
    assert set(sub['Predicted'].unique()).issubset({0,1})

sub_safe.to_csv('submission_v13_hybrid_safe.csv',index=False)
sub_routed.to_csv('submission_v13_segment_routed.csv',index=False)

report={'version':'V13_HYBRID_STANDALONE','python':sys.version.split()[0],'pandas':pd.__version__,'train_rows':len(train),'test_rows':len(test),'train_content_coverage':float(train_has_content.mean()),'test_content_coverage':float(test_has_content.mean()),'url_oof_f1':float(url_f1),'hybrid_oof_f1':float(hyb_f1),'url_covered_f1':float(url_f1_cov),'hybrid_covered_f1':float(hyb_f1_cov),'use_hybrid_global':USE_HYBRID_GLOBAL,'use_hybrid_routed':USE_HYBRID_ROUTED}
pd.DataFrame([report]).to_csv('v13_validation_report.csv',index=False)
with open('v13_final_bundle.pkl','wb') as f:
    pickle.dump({'report':report,'url_meta_final':url_final,'hybrid_meta_final':hyb_final,'url_threshold':url_thr,'hybrid_threshold':hyb_thr,'hybrid_covered_threshold':hyb_thr_cov,'base_names':BASE_NAMES,'exact_lookup':exact_lookup},f,pickle.HIGHEST_PROTOCOL)
print('Saved submissions/report/bundle')
display(pd.DataFrame([report]))


## Как выбирать итог

- Если `USE_HYBRID_GLOBAL=True`, первый кандидат — `submission_v13_hybrid_safe.csv`.
- Если `USE_HYBRID_ROUTED=True`, второй кандидат — `submission_v13_segment_routed.csv`.
- Если оба флага `False`, content не доказал пользу и notebook сознательно оставляет URL-only prediction вместо ухудшения результата.

Смотрите на F1/precision/recall, а не на долю единиц в submission.
